# Kathmandu traffic vision - plate reader run 2

Run 2 of the plate reader's **character classifier**, with two more Nepali character sets (Apache-2.0 and MIT; 2,933 and 4,410 crops). First the run-1 classifier, never trained on them, labels every crop: an honest test on unseen sources, and a check of the romanised folder names (ba, pa, cha ...); folders whose labels disagree are dropped. Then the classifier is retrained on all sources and scored per source. The character detector is unchanged (no new plates with character boxes).

In [ ]:
!pip install -q ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:

import os, random, yaml
from pathlib import Path
IN, OUT = Path("/kaggle/input/datasets"), Path("/kaggle/working/data")
VAL_LISTS = {}

def label_of(img):
    parts = list(img.parts)
    if "images" in parts:                                       # YOLO layout: .../images/... -> .../labels/...
        i = len(parts) - 1 - parts[::-1].index("images")
        p = Path(*parts[:i], "labels", *parts[i + 1:]).with_suffix(".txt")
        if p.exists():
            return p
    return img.with_suffix(".txt")                              # some sets keep labels next to images

def add(tag, split, images, names, mapping, repeat=1):
    """Link images into OUT/<split>, rewriting labels: keep only source classes in `mapping`
    (source name -> target id); boxes of other classes are dropped, the image stays."""
    d = OUT / split
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)
    n_box, kept = 0, []
    for img in images:
        rows, lab = [], label_of(img)
        if lab.exists():
            for ln in lab.read_text(errors="ignore").splitlines():
                parts = ln.split()
                if len(parts) != 5 or not parts[0].isdigit():      # skip polygons / malformed rows
                    continue
                k = int(parts[0])
                if k < len(names) and names[k] in mapping:
                    rows.append(" ".join([str(mapping[names[k]])] + parts[1:]))
        for r in range(repeat):
            stem = f"{tag}_{r}_{img.parent.name}_{img.stem}".replace(" ", "_")
            link = d / "images" / (stem + img.suffix.lower())
            if not link.exists():
                link.symlink_to(img)
            (d / "labels" / (stem + ".txt")).write_text("\n".join(rows))
            kept.append(str(link))
        n_box += len(rows)
    if split == "val":
        VAL_LISTS[tag] = kept
    print(f"{tag:<10} {split:<5} {len(images):>6} images x{repeat}  {n_box:>7} boxes kept")

def imgs(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

def split_random(images, val_share=0.15, seed=0):
    images = images[:]
    random.Random(seed).shuffle(images)
    n = int(len(images) * val_share)
    return images[n:], images[:n]

def write_yaml(names):
    (OUT / "data.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images", "val": "val/images",
                                                 "names": dict(enumerate(names))}))
    for tag, files in VAL_LISTS.items():          # per-source validation sets, to see where the model is weak
        (OUT / f"val_{tag}.txt").write_text("\n".join(files))
        (OUT / f"data_{tag}.yaml").write_text(yaml.safe_dump({"path": str(OUT), "train": "train/images",
                                                            "val": f"val_{tag}.txt", "names": dict(enumerate(names))}))


In [ ]:

import random, shutil, glob
from collections import Counter
from pathlib import Path
from ultralytics import YOLO
CLS = Path("/kaggle/working/glyphs")
VAL = {}     # source -> [(image link, class)]

def link_all(tag, pairs, split):
    for i, (img, c) in enumerate(pairs):
        d = CLS / split / c
        d.mkdir(parents=True, exist_ok=True)
        dst = d / f"{tag}_{i}{img.suffix.lower()}"
        if not dst.exists():
            dst.symlink_to(img)
        if split == "val":
            VAL.setdefault(tag, []).append((dst, c))
    print(f"{tag:<10} {split:<5} {len(pairs):>6} crops, {len(set(c for _, c in pairs))} classes")

# Devanagari: one folder per character, no split given -> random 15% validation
dev = IN / "inspiring-lab/nepali-number-plate-characters-dataset/character_ocr"
pairs = [(p, p.parent.name) for p in imgs(dev)]
random.Random(0).shuffle(pairs)
n = int(len(pairs) * 0.15)
link_all("devanagari", pairs[n:], "train")
link_all("devanagari", pairs[:n], "val")

# Embossed: class folders under train/ (2,670), valid/ (349) and test/ (563); the flat augmented/ folder
# (6,566 files, no class in path or name) is skipped. utsavbaral007's set has the same test folders: a copy, not used.
emb = IN / "theciceerguy/embossed-number-plate-detection-nepal-dataset/digits_and_numbers_dataset"
name = lambda p: "flag" if p.parent.name == "Nepali Flag" else "emb_" + p.parent.name
link_all("embossed", [(p, name(p)) for p in imgs(emb / "train")], "train")
link_all("embossed", [(p, name(p)) for p in imgs(emb / "valid") + imgs(emb / "test")], "val")


from collections import Counter
ROMAN = {**{str(i): "०१२३४५६७८९"[i] for i in range(10)},
         "ba": "बा", "pa": "प", "cha": "च", "kha": "ख", "ko": "को", "me": "मे", "pra": "प्र"}
NEW = {"ishwor_chars": IN / "ishworsubedii/nepalese-license-plate-character-dataset/dataset",
       "kshitiz_chars": IN / "kshitizgajurel042/individual-character-images-of-nepali-lp/NepaliLPCharacters"}
run1 = YOLO(sorted(glob.glob("/kaggle/input/**/glyphs_best.pt", recursive=True))[0])
unseen = []
for tag, root in NEW.items():
    ok_n = all_n = 0
    pairs = []
    for folder in sorted(p for p in root.iterdir() if p.is_dir()):
        files = imgs(folder)
        want = ROMAN.get(folder.name)
        got = Counter()
        for i in range(0, len(files), 512):
            for r in run1.predict([str(f) for f in files[i:i + 512]], imgsz=64, verbose=False):
                got[r.names[r.probs.top1]] += 1
        top, n = got.most_common(1)[0]
        hit = got.get(want, 0)
        print(f"{tag:<14} {folder.name:<4} -> {want}  {len(files):>4} crops, run-1 says {top} for {n}, {hit / len(files):.1%} agree")
        if want is not None:            # every mapped folder counts, kept or dropped, so the score is not flattered
            ok_n += hit
            all_n += len(files)
        if want is None or top != want:
            print(f"   dropped: mapping not confirmed")
            continue
        pairs += [(f, want) for f in files]
    unseen.append({"source": tag, "crops": all_n, "run1_top1_unseen": round(ok_n / all_n, 4)})
    random.Random(0).shuffle(pairs)
    n = int(len(pairs) * 0.15)
    link_all(tag, pairs[n:], "train")
    link_all(tag, pairs[:n], "val")
import pandas as pd
pd.DataFrame(unseen).to_csv("/kaggle/working/run1_on_unseen_sources.csv", index=False)
print(pd.DataFrame(unseen).to_string(index=False))

cls = YOLO("yolo11s-cls.pt")
cls.train(data=str(CLS), epochs=40, time=1.5, imgsz=64, batch=256, patience=10,
          project="/kaggle/working/runs", name="glyphs", workers=4, seed=0, plots=True)
shutil.copy(sorted(glob.glob("/kaggle/working/runs/glyphs*/weights/best.pt"))[-1], "/kaggle/working/glyphs_best.pt")

m = YOLO("/kaggle/working/glyphs_best.pt")
rows = []
for tag, items in VAL.items():
    wrong = Counter()
    ok = 0
    for i in range(0, len(items), 512):
        batch = items[i:i + 512]
        for (img, c), r in zip(batch, m.predict([str(p) for p, _ in batch], imgsz=64, verbose=False)):
            got = r.names[r.probs.top1]
            ok += got == c
            if got != c:
                wrong[f"{c} -> {got}"] += 1
    rows.append({"source": tag, "crops": len(items), "top1": round(ok / len(items), 4),
                 "most_confused": "; ".join(f"{k} ({v})" for k, v in wrong.most_common(5))})
import pandas as pd
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/glyphs_per_source.csv", index=False)
print(res.to_string(index=False))
shutil.rmtree(CLS)
